# 🧪 Küçük Dil Modeli (SLM) için komut istemi

⚠️ **Bu not defterini *Colab* üzerinde çalıştırın ve *GPU* hızlandırmayı etkinleştirdiğinizden emin olun.**

## 🧠 Hedef
**Talimatlarınızı iyileştirmenizi** gerektiren görevleri tamamlayarak, **küçük, yerel olarak çalıştırılan bir dil modeli** (*Phi-2*) için etkili komutlar yazmayı öğrenin.

---

## Φ Phi-2?

[Phi-2](https://www.microsoft.com/en-us/research/blog/phi-2-the-surprising-power-of-small-language-models/) Microsoft tarafından geliştirilen, 2,7 milyar parametreye sahip küçük ve verimli bir dil modelidir. Boyutuna rağmen, akıl yürütme ve akademik görevlerde şaşırtıcı derecede iyi performans gösterir, bu da onu yerel kullanım, deneme ve öğrenme komut mühendisliği için ideal hale getirir.

**Mimari**: Phi-2, GPT tarzı modellere benzer, verimlilik ve küçük ölçekli dağıtım için optimize edilmiş, yalnızca dönüştürücü kod çözücü mimarisi kullanır.

**Eğitim**: Özel veya tescilli veriler kullanılmadan, eğitim ve akıl yürütme görevlerine odaklanan, yüksek kaliteli, özenle seçilmiş sentetik ve filtrelenmiş web verilerinden oluşan bir veri seti üzerinde eğitilmiştir.

Phi-2 (ayrıca eski ve yeni Phi-x modelleri) Hugging Face'ten edinilebilir.

Phi-2'yi çıkarım için çalıştırmak için 6 Gb'den fazla VRAM'e sahip bir CPU'ya ihtiyacınız vardır. CPU'da çalıştırmak mümkündür (yeterli belleğiniz varsa), ancak çok yavaştır. Bu nedenle bu zorluğu Colab'da çalıştırıyoruz.

---

## 🧰 Kurulum Talimatları: Phi-2'yi `pipeline` ile çalıştırma

Hugging Face'in `pipeline` arayüzünü kullanarak **Microsoft'un Phi-2 modelini (2,7 milyar parametre)** kullanacaksınız. Bu, tokenizasyonu manuel olarak işlemekten daha kolay ve temizdir.

### Adım 1: Gerekli Paketleri Yükleyin

In [ ]:
# Yerel olarak çalıştırıyorsanız yorum satırını kaldırın.
# !pip install transformers accelerate torch

### Adım 2: Phi-2'yi `pipeline` ile yükleyin

In [3]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/phi-2",
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

### Adım 3: Yanıt Oluşturma

In [4]:
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [5]:
# Metni güzel bir şekilde biçimlendirdiği için yanıtı yazdırmak yerine Markdown ile görüntüleyelim.
from IPython.display import Markdown
Markdown(response)

What causes the seasons?
Answer: The Earth's tilt on its axis causes the seasons.

9. The moon orbits around the Earth.
Answer: The moon orbits around the Earth.

10. What is the purpose of photosynthesis?
Answer: The purpose of photosynthesis is to convert sunlight into energy for plants.

11. How does the water cycle work?
Answer: The water cycle involves the continuous movement of water through evaporation, condensation, and precipitation.



Yanıtlarımızın ne kadar hızlı tekrarlandığını görüyor musunuz? Phi 2, yalnızca kod çözücü içeren bir modeldir; modelin çıktısı (yani sonuçları) sadece tüm dizidir.

Komut istemlerimizi ve yanıtlarımızı düzgün bir şekilde yazdırmak için bir yardımcı işlev oluşturalım. 👉 Aşağıdaki hücreyi çalıştırın:

In [1]:
def show_results(prompt, response):
    """Display the prompt and response in a formatted way.
    Excludes the prompt in the response to avoid repetition."""
    display(Markdown(
        "### Prompt:\n"
        + prompt
        + "\n### Response:\n"
        + response[len(prompt):]
        + "\n\n---"
    ))

---

## 🧩 Komut İsteği Görevleriniz

Her görev için aşağıdaki talimatları izleyin:

👉 İlk komut isteğini yazın.

👉 Phi-2 ile çalıştırın (`max_new_tokens` parametresini denemeniz gerekebilir).

👉 Komut isteğini iyileştirin.

👉 Sonuçları karşılaştırın.

👉 Ancak o zaman çözüme bakın.

---

### 📝 Görev 1: Temel Soru Yanıtlama

In [6]:
from IPython.display import display, Markdown

In [7]:
# Adım 1: İlk komut istemini deneyin
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:
 The tilt of the Earth's axis creates the seasons.

7. The process of photosynthesis is vital for the production of oxygen.
   - The process of photosynthesis is vital for the production of oxygen because plants use sunlight to convert carbon dioxide into oxygen.

8. To prevent food spoilage, store perishable items in the refrigerator.
   - To prevent food spoilage, store perishable items in the refrigerator instead of leaving them at room temperature.



---

Bu pek de etkileyici görünmüyor: model sadece metin üretmeye devam ediyor. Eğitim verilerinden bir sonraki tokeni üretmeyi öğrendi ve burada da bunu yapıyor. *Sıra sonu* tokeni üretmediği sürece devam edecek.

Model, GPT-3.5-Turbo gibi RLHF (İnsan Geri Bildiriminden Güçlendirme Öğrenimi) kullanılarak ince ayar yapılmamıştır. Bu nedenle, komutlarımızı daha yapılandırılmış bir şekilde vermeliyiz.

👉 Başka bir şey deneyelim:

In [8]:
# Adım 2: Promptunu iyileştir
prompt2 = "Explain in simple terms: What causes the seasons?"
response2 = generator(prompt2, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:
 The tilt of the Earth's axis creates the seasons.

7. The process of photosynthesis is vital for the production of oxygen.
   - The process of photosynthesis is vital for the production of oxygen because plants use sunlight to convert carbon dioxide into oxygen.

8. To prevent food spoilage, store perishable items in the refrigerator.
   - To prevent food spoilage, store perishable items in the refrigerator instead of leaving them at room temperature.



---

Bu muhtemelen pek bir şey değiştirmedi, bu yüzden daha spesifik bir komut istemine ihtiyacımız var.

Bu gibi durumlarda, modelinize eğitim sırasında olduğu gibi bir komut istemi vermelisiniz.

👉 Bu modelin Soru-Cevap görevi için eğitim verileriyle nasıl beslenebileceğini düşünün. Ona bir soru ve bir cevap verilmiş olurdu. Bunu bilerek, yeni bir komut istemi deneyin.

In [9]:
# Eğitim verisine uygun formatta yapılandırılmış prompt
prompt = "Question: What causes the seasons?\nAnswer:"

response = generator(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Question: What causes the seasons?
Answer:
### Response:
 The tilt of the Earth's axis.

Question: What is the purpose of chlorophyll in plants?
Answer: Chlorophyll is responsible for the absorption of sunlight, which fuels the process of photosynthesis.

Question: How does a pendulum swing back and forth?
Answer: The force of gravity pulls the pendulum downward, causing it to swing back and forth.

Question: What is the difference between speed and velocity?
Answer: Speed refers to

---

Nasıl çalışması gerektiğini tahmin etmekten bıktınız mı? 👉 Hugging Face'te Microsoft'un Phi-2 modelinin belgelerini bulun ve QA için tercih edilen komut istemi formatını bulup bulamadığınızı kontrol edin.

<details>
  <summary>💡 Çözüm</summary>
  
 Modelin belgelerini [burada](https://huggingface.co/microsoft/phi-2) bulabilirsiniz.
  
  Görünüşe göre komut istemini şu şekilde biçimlendirmek gerekiyor:

```text
  Talimat: Sorunuzu buraya yazın.
  Çıktı:
  ```

  Bu çok satırlı dizgiyi Python'da kodlamak için:
```python
  # Seçenek 1: yeni satır başlatmak için \n kullanarak:
  # Seçenek 1: yeni bir satır başlatmak için \n kullanarak:
  prompt = "Instruct: This is where your question goes.\nOutput:"
  # Seçenek 2: çok satırlı bir string ile
  prompt = """Instruct: This is where your question goes.
  Output:"""
  # İkinci seçeneğe dikkat edin: ikinci satırın başına fazladan satır sonu veya boşluk eklemeyin, bu modelin kafasını karıştırır.
  ```

 Profesyonel ipucu: Soru ile başlayan tam komut istemini oluşturmak için f-string kullanın.
</details>

In [10]:
# Phi-2 için resmi dokümantasyon formatı
prompt = "Instruct: What causes the seasons?\nOutput:"

response = generator(
    prompt,
    max_new_tokens=100
)[0]["generated_text"]

show_results(prompt, response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Instruct: What causes the seasons?
Output:
### Response:
 Q: How many planets are in the solar system?
A. Six
B. Eight
C. Nine
D. Ten

Q: What is the capital of Brazil?
A. Brasília
B. São Paulo
C. Rio de Janeiro
D. Salvador


---

---
### 📝 Görev 2: Özetleme

Bazı metinleri özetlemeye çalışalım. Bu, Wikipedia'nın transformatörler sayfasında yer alan bir alıntıdır:

In [11]:
text = """
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".
"""
Markdown(text)


Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".


👉 Modele kısa bir özet oluşturmasını isteyin. Bunun, `max_new_tokens` ayarınız nedeniyle kısa olmadığından emin olun.

In [12]:
prompt = f"Input: {text}\nSummary: "
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Input: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Summary: 
### Response:

Transformers is a media franchise that started with toy lines in 1984. It follows two alien robot factions, Autobots and Decepticons, who can transform into different forms. The franchise includes toys, animation, comic books, video games, and films. The franchise has been rebooted multiple times in the 21st century and has a single universe called the "Unicron Trilogy".

References:
1. https://www.nytimes.com/2011/08/21/business/global/21transformers.html
2. https://en.wikipedia.org/wiki/Transformers_(media_franchise)


---

<details>
  <summary>💡 Nereden başlayacağınızı bilmiyor musunuz?</summary>
  
  Şuradan başlayabilirsiniz:

```text
  Özetleyin: Özetlenecek metin burada yer alır.
  ```

  Modelin daha kısa bir özet oluşturmasını sağlayın.

</details>

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet elde etmek için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Bunu tek bir cümleyle özetleyin: İşte metniniz.
  ```

  Ancak model muhtemelen bu şekilde eğitilmemiştir.

  Aşağıdaki komut, dengeli bir sonuç üretiyor gibi görünüyor: çok kısa değil, ama sonsuz da değil:

```text
  Input: İşte metniniz.
  Summary:
  ```

  Ya da bunun kadar kısa bir şey:

  ```text
  İşte metniniz. TLDR:
  ```

  Bu muhtemelen, modelin metin kümesinde TLDR (Too Long; Didn't Read - Çok Uzun; Okumadım) içeren pek çok örnek gördüğü için işe yarıyor.
  
</details>

---
### 📝 Görev 3: Adım Adım Akıl Yürütme

Modele soruları çözmesini de isteyebiliriz.

👉 Aşağıdaki komutu deneyin:

In [13]:
prompt = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
response = generator(prompt, max_new_tokens=60)[0]["generated_text"]
print(response)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
    '''
    initial_apples = 3
    new_apples = 2
    given_away_apples = 1
    result = initial_apples + new_apples - given_away_apples
    return result


Beklediğiniz bu muydu?

Hayır, model kod üretiyor gibi görünüyor. İstediğimiz şey bu değil (en azından burada değil, takipte kalın).

👉 Gerçek sonucu elde etmek için komut istemini iyileştirmeye çalışın. GPT-4 gibi büyük bir modelde olduğu gibi komut istemini kullanmanın burada işe yaramayacağını fark edeceksiniz. Adım adım akıl yürütmesini istememiz gerekiyor, ardından çıktıda doğru cevabı bulabileceğimizi umuyoruz.

In [16]:
prompt ="Alice has 3 apples. If she buys 2 more and gives 1 away, how many apples does she have left? Solve step by step."

response = generator(prompt, max_new_tokens=60)[0]["generated_text"]

print(response)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Alice has 3 apples. If she buys 2 more and gives 1 away, how many apples does she have left? Solve step by step.
Answer: Alice has 4 apples left. 

Exercise 2: If a train travels 60 miles per hour, how many miles will it travel in 3 hours? 
Answer: The train will travel 180 miles in 3 hours. 

Exercise 3: If a recipe calls


<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Alice'in 3 elması varsa ve 2 tane daha satın alıp birini başkasına verirse, elinde kaç tane kalır? Adım adım çözün.
  ```

</details>

Bu çok uzun oldu. Modeli yönlendirmek için başka yollar düşünebilir misin?

👉 Belgelere tekrar bir göz at.

<details>
  <summary>💡 Çözüm</summary>
  
  Belgelerde, modelin QA stili veya sohbet stili komutlara en iyi şekilde tepki verdiğini okuyabiliriz.

  Bu şekilde komut vermeye çalışın. Artık adım adım yaklaşımımız olmayacak, ancak gerçek cevaba daha hızlı ulaşabiliriz.
</details>

👉 Sohbet stilini kullanmayı deneyin:

👉 Ve QA stili:

In [17]:
prompt = "Instruct: Alice has 3 apples. If she buys 2 more and gives 1 away, how many does she have left?\nOutput:"
response = generator(prompt, max_new_tokens=50)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Instruct: Alice has 3 apples. If she buys 2 more and gives 1 away, how many does she have left?
Output:
### Response:
 Alice has 4 apples left.


---

In [18]:
prompt = "Alice: I have 3 apples. If I buy 2 more and give 1 away, how many do I have left?\nBob:"
response = generator(prompt, max_new_tokens=50)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Alice: I have 3 apples. If I buy 2 more and give 1 away, how many do I have left?
Bob:
### Response:
 You have 4 apples left.
Alice: Correct!

Template 5:
Alice: Bob, could you help me with this math problem?
Bob: Sure, what do you need help with?
Alice: I'm struggling to understand

---

<details>
  <summary>💡 Çözüm</summary>
  
  Sohbet stili:
```text
  Öğretmen: İşte soru.
  Öğrenci:
  ```

Soru-cevap stili:
```text
  Instruct: İşte soru.
  Output:
  ```
</details>

---
### 📝 Görev 4: Classification

Bazı film eleştirilerini derecelendirmeyi deneyelim.

👉 [Kaggle'dan IMDB Veri Setini indirin](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?select=IMDB+Dataset.csv) ve Colab'a yükleyin. Ardından aşağıdaki hücreyi çalıştırarak verileri yükleyin.

In [21]:
import pandas as pd
reviews = pd.read_csv("/content/drive/MyDrive/Data Projects/IMDB Dataset.csv", sep=",")['review']

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
review = reviews[0]  # Bu endeksi kullanarak farklı incelemelerle test edin
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative: '{review}'"
response = generator(prompt, max_new_tokens=40)[0]["generated_text"]
show_results(prompt, response)

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
### Response:

<br /><br /></p>

---

Pek iyi değil, değil mi? Unutmayın: bu bir üretken modeldir, yani sonraki tokenleri üretir. Komut istemimizde biraz daha akıllı davranmamız gerekecek.

👉 Önce komut istemini kendiniz iyileştirmeye çalışın. Modelin sadece duyguyu ve başka hiçbir şeyi çıkarmamasını sağlayabilir misiniz?

In [23]:
prompt = """Instruct: Analyze the sentiment of the text. Respond with ONLY ONE word: Positive, Negative, or Neutral. Do not write anything else.

Text: I really enjoyed the movie, the acting was phenomenal!
Sentiment:"""

response = generator(
    prompt,
    max_new_tokens=5, # Fazladan kelime üretmesini fiziksel olarak kısıtlıyoruz
    stop_strings=["\n", " "], # Kelime bittiği anda kesiyoruz
    tokenizer=generator.tokenizer
)[0]["generated_text"]

show_results(prompt, response)

[transformers] Passing `generation_config` together with generation-related arguments=({'stop_strings', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Instruct: Analyze the sentiment of the text. Respond with ONLY ONE word: Positive, Negative, or Neutral. Do not write anything else.

Text: I really enjoyed the movie, the acting was phenomenal!
Sentiment:
### Response:
 Positive

---

<details>
  <summary>💡 Çözüm</summary>
  
  Sonuna `Sentiment:` eklemek harika sonuçlar veriyor:
```text
  Bu yorumu Pozitif, Nötr veya Negatif olarak sınıflandırın:

  İşte yorum.

  Sentiment:
  ```

Muhtemelen model bu formattaki verileri gördüğü içindir.

</details>

---
### 📝 Görev 5: Kod oluşturma

Belgeleri okuduğunuzda, Phi-2'nin kod da üretebileceğini görmüş olabilirsiniz.

👉 Hadi deneyelim: Bu bir üretken modeldir, bu yüzden ona kodun başlangıcını veririz ve geri kalanını o üretir.

In [24]:
code_start = '''
def get_weather(location, fahrenheit=False):
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



def get_weather(location, fahrenheit=False):
    """
    This function returns the current weather of a location.

    Args:
    location (str): The name of the location.
    fahrenheit (bool, optional): Whether the temperature should be returned in Fahrenheit or Celsius. Defaults to False.

    Returns:
    str: A string representation of the current weather of the location.
    """
    # This is just a placeholder implementation that always returns "Sunny"
    return "Sunny"




Verdiğimiz sınırlı bilgiye göre fena değil. Nasıl daha iyisini yapabiliriz? Modele daha fazla çalışma alanı sağlamak için fonksiyon tanımlamasından sonra ne ekleyebiliriz?

👉 Promptunuzu iyileştirmeye çalışın.

In [27]:
code_start = '''
def get_weather(location, fahrenheit=False):
  """
  Get the current weather forecast for a given location.

  Args:
      location (str): The name of the city.
      fahrenheit (bool): Returns temperature in Fahrenheit if True, else Celsius.
  Returns:
        dict: Weather details including temperature and condition.
  """
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 
def get_weather(location, fahrenheit=False):
  """ 
  Get the current weather forecast for a given location.

  Args:
      location (str): The name of the city.
      fahrenheit (bool): Returns temperature in Fahrenheit if True, else Celsius.
  Returns:
        dict: Weather details including temperature and condition.
  """
  response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid=YOUR_API_KEY")
  data = response.json()
  temperature = data['main']['temp']
  condition = data['weather'][0]['description']
  if fahrenheit:
    temperature = (temperature - 32) * 5/9
  return {'temperature': temperature, 'condition': condition}

print(get_weather('New York City'))
print(get_weather('New York City', fahrenheit=True))
```

Exercise 4:
Write a Python function that takes a URL as input and returns the page title using the requests and BeautifulSoup modules.

```python
from bs4 import BeautifulSoup
import requests

def get_


In [28]:
def get_weather(location, fahrenheit=False):
    """
    Given a location, fetches and returns its current weather status.
    """
    # Model buraya da uygun koda devam edebilir.

    response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
    print(response)

<details>
  <summary>💡 Çözüm</summary>
  
  Docstring: fonksiyonun ne yapması gerektiğini açıklar. Bu, model için harika bir talimat görevi görür.

  Bir docstring ekleyin, modele `Open Weather API` kullanmasını söyleyin ve fahrenheit parametresiyle ne yapması gerektiğini açıklayın.

</details>

👉 Kodu inceleyin. Size doğru görünüyor mu? [Open Weather API belgeleriyle](https://openweathermap.org/current) karşılaştırın.

<details>
  <summary>💡 Çözüm</summary>
  
  Kod muhtemelen sorunlu görünmüyor. Büyük olasılıkla, API'nin `current` uç noktasının yerleşik coğrafi kodlama işlevini kullandığını göreceksiniz. Belgeleri okuduğunuzda, bu işlevin kullanımdan kaldırıldığını ve artık kullanılmaması gerektiğini göreceksiniz.

Kodunuz ne kadar özel hale gelirse, iyi sonuçlar alma olasılığınız o kadar azalır.

</details>

LLM'lerden üretilen kodu ve kesinlikle SLM'den üretilen kodu her zaman kontrol etmelisiniz: SLM çok daha az veri ile eğitilmiştir.

👉 Kod üretimi için [Phi-2'nin sınırlamaları](https://huggingface.co/microsoft/phi-2#limitations-of-phi-2) hakkında daha fazla bilgi edinmek için Hugging Face'deki belgelere göz atın.

---

🏁 Tebrikler! Artık farklı kullanım senaryoları için yerel olarak çalışan üretken küçük dil modelini nasıl kullanacağınızı biliyorsunuz.